# 🧠 RNN 학습 코드 (주석으로 완전 해설)

---

## 🎯 목표
입력: [1, 2, 3]  
출력: [2, 3, 4]  

👉 "+1 규칙"을 RNN이 학습하도록 만든다

---

## 📌 전체 코드 (Forward + Loss + Backprop)

```python
import numpy as np

# =========================
# 1️⃣ 데이터 준비
# =========================
x = np.array([1, 2, 3])   # 입력 시퀀스
y = np.array([2, 3, 4])   # 정답 (한 칸씩 증가)

# =========================
# 2️⃣ 가중치 초기화 (직접 지정)
# =========================
Wx = 0.5   # 입력 → hidden 가중치
Wh = 0.8   # hidden → hidden 가중치 (기억)
Wy = 1.0   # hidden → 출력 가중치

bh = 0.0   # hidden bias
by = 0.0   # output bias

lr = 0.01  # 학습률

# =========================
# 3️⃣ 학습 반복
# =========================
for epoch in range(1):  # 이해용이라 1번만 실행

    # hidden state 초기화 (기억 시작점)
    h = 0

    # 각 시점의 hidden과 출력 저장
    hs = []
    ys = []

    # =========================
    # 4️⃣ Forward (순전파)
    # =========================
    for t in range(len(x)):

        # 🔹 RNN 핵심 공식
        # h_t = tanh(Wx * x_t + Wh * h_{t-1} + b)
        h = np.tanh(Wx * x[t] + Wh * h + bh)

        # 🔹 출력 계산
        # y_hat = Wy * h + by
        y_hat = Wy * h + by

        # 값 저장 (나중에 역전파에서 사용)
        hs.append(h)
        ys.append(y_hat)

        # 👉 중간 계산 출력
        print(f"[Forward] t={t}")
        print(f"  input x = {x[t]}")
        print(f"  hidden h = {h:.3f}")
        print(f"  prediction y_hat = {y_hat:.3f}")
        print()

    # =========================
    # 5️⃣ Loss 계산 (MSE)
    # =========================
    loss = np.mean((np.array(ys) - y) ** 2)

    print("===== LOSS =====")
    print("predictions:", ys)
    print("targets    :", y)
    print(f"loss = {loss:.3f}")
    print()

    # =========================
    # 6️⃣ Backprop (역전파)
    # =========================
    # 기울기 초기화
    dWx = 0
    dWh = 0
    dWy = 0
    dbh = 0
    dby = 0

    dh_next = 0  # 다음 시점에서 넘어오는 gradient

    # 🔥 시간 역순으로 진행 (BPTT)
    for t in reversed(range(len(x))):

        print(f"[Backward] t={t}")

        # 🔹 출력 오차
        # dy = dLoss/dy_hat
        dy = 2 * (ys[t] - y[t])
        print(f"  dy = {dy:.3f}")

        # 🔹 Wy gradient
        dWy += dy * hs[t]
        dby += dy

        print(f"  dWy += {dy:.3f} * {hs[t]:.3f}")

        # 🔹 hidden으로 gradient 전달
        # dh = 현재 오차 + 미래에서 온 gradient
        dh = dy * Wy + dh_next
        print(f"  dh = {dh:.3f}")

        # 🔹 tanh 미분
        # dtanh = (1 - h^2) * dh
        dtanh = (1 - hs[t]**2) * dh
        print(f"  dtanh = {dtanh:.3f}")

        # 🔹 Wx gradient
        dWx += dtanh * x[t]

        # 🔹 Wh gradient (이전 hidden 사용)
        prev_h = hs[t-1] if t > 0 else 0
        dWh += dtanh * prev_h

        dbh += dtanh

        print(f"  dWx += {dtanh:.3f} * {x[t]}")
        print(f"  dWh += {dtanh:.3f} * {prev_h:.3f}")

        # 🔹 다음 step으로 gradient 전달
        dh_next = dtanh * Wh

        print(f"  dh_next = {dh_next:.3f}")
        print()

    # =========================
    # 7️⃣ 가중치 업데이트
    # =========================
    Wx -= lr * dWx
    Wh -= lr * dWh
    Wy -= lr * dWy
    bh -= lr * dbh
    by -= lr * dby

    print("===== UPDATE =====")
    print(f"Wx = {Wx:.3f}")
    print(f"Wh = {Wh:.3f}")
    print(f"Wy = {Wy:.3f}")

In [1]:
# 🧠 RNN 학습 코드 (주석으로 완전 해설)

# ---

# ## 🎯 목표
# 입력: [1, 2, 3]  
# 출력: [2, 3, 4]  

# 👉 "+1 규칙"을 RNN이 학습하도록 만든다

# ---

# ## 📌 전체 코드 (Forward + Loss + Backprop)

# ```python
import numpy as np

# =========================
# 1️⃣ 데이터 준비
# =========================
x = np.array([1, 2, 3])   # 입력 시퀀스
y = np.array([2, 3, 4])   # 정답 (한 칸씩 증가)

# =========================
# 2️⃣ 가중치 초기화 (직접 지정)
# =========================
Wx = 0.5   # 입력 → hidden 가중치
Wh = 0.8   # hidden → hidden 가중치 (기억)
Wy = 1.0   # hidden → 출력 가중치

bh = 0.0   # hidden bias
by = 0.0   # output bias

lr = 0.01  # 학습률

# =========================
# 3️⃣ 학습 반복
# =========================
for epoch in range(1):  # 이해용이라 1번만 실행

    # hidden state 초기화 (기억 시작점)
    h = 0

    # 각 시점의 hidden과 출력 저장
    hs = []
    ys = []

    # =========================
    # 4️⃣ Forward (순전파)
    # =========================
    for t in range(len(x)):

        # 🔹 RNN 핵심 공식
        # h_t = tanh(Wx * x_t + Wh * h_{t-1} + b)
        h = np.tanh(Wx * x[t] + Wh * h + bh)

        # 🔹 출력 계산
        # y_hat = Wy * h + by
        y_hat = Wy * h + by

        # 값 저장 (나중에 역전파에서 사용)
        hs.append(h)
        ys.append(y_hat)

        # 👉 중간 계산 출력
        print(f"[Forward] t={t}")
        print(f"  input x = {x[t]}")
        print(f"  hidden h = {h:.3f}")
        print(f"  prediction y_hat = {y_hat:.3f}")
        print()

    # =========================
    # 5️⃣ Loss 계산 (MSE)
    # =========================
    loss = np.mean((np.array(ys) - y) ** 2)

    print("===== LOSS =====")
    print("predictions:", ys)
    print("targets    :", y)
    print(f"loss = {loss:.3f}")
    print()

    # =========================
    # 6️⃣ Backprop (역전파)
    # =========================
    # 기울기 초기화
    dWx = 0
    dWh = 0
    dWy = 0
    dbh = 0
    dby = 0

    dh_next = 0  # 다음 시점에서 넘어오는 gradient

    # 🔥 시간 역순으로 진행 (BPTT)
    for t in reversed(range(len(x))):

        print(f"[Backward] t={t}")

        # 🔹 출력 오차
        # dy = dLoss/dy_hat
        dy = 2 * (ys[t] - y[t])
        print(f"  dy = {dy:.3f}")

        # 🔹 Wy gradient
        dWy += dy * hs[t]
        dby += dy

        print(f"  dWy += {dy:.3f} * {hs[t]:.3f}")

        # 🔹 hidden으로 gradient 전달
        # dh = 현재 오차 + 미래에서 온 gradient
        dh = dy * Wy + dh_next
        print(f"  dh = {dh:.3f}")

        # 🔹 tanh 미분
        # dtanh = (1 - h^2) * dh
        dtanh = (1 - hs[t]**2) * dh
        print(f"  dtanh = {dtanh:.3f}")

        # 🔹 Wx gradient
        dWx += dtanh * x[t]

        # 🔹 Wh gradient (이전 hidden 사용)
        prev_h = hs[t-1] if t > 0 else 0
        dWh += dtanh * prev_h

        dbh += dtanh

        print(f"  dWx += {dtanh:.3f} * {x[t]}")
        print(f"  dWh += {dtanh:.3f} * {prev_h:.3f}")

        # 🔹 다음 step으로 gradient 전달
        dh_next = dtanh * Wh

        print(f"  dh_next = {dh_next:.3f}")
        print()

    # =========================
    # 7️⃣ 가중치 업데이트
    # =========================
    Wx -= lr * dWx
    Wh -= lr * dWh
    Wy -= lr * dWy
    bh -= lr * dbh
    by -= lr * dby

    print("===== UPDATE =====")
    print(f"Wx = {Wx:.3f}")
    print(f"Wh = {Wh:.3f}")
    print(f"Wy = {Wy:.3f}")

[Forward] t=0
  input x = 1
  hidden h = 0.462
  prediction y_hat = 0.462

[Forward] t=1
  input x = 2
  hidden h = 0.879
  prediction y_hat = 0.879

[Forward] t=2
  input x = 3
  hidden h = 0.976
  prediction y_hat = 0.976

===== LOSS =====
predictions: [np.float64(0.46211715726000974), np.float64(0.8786223746838898), np.float64(0.9758816208890569)]
targets    : [2 3 4]
loss = 5.337

[Backward] t=2
  dy = -6.048
  dWy += -6.048 * 0.976
  dh = -6.048
  dtanh = -0.288
  dWx += -0.288 * 3
  dWh += -0.288 * 0.879
  dh_next = -0.231

[Backward] t=1
  dy = -4.243
  dWy += -4.243 * 0.879
  dh = -4.473
  dtanh = -1.020
  dWx += -1.020 * 2
  dWh += -1.020 * 0.462
  dh_next = -0.816

[Backward] t=0
  dy = -3.076
  dWy += -3.076 * 0.462
  dh = -3.892
  dtanh = -3.061
  dWx += -3.061 * 1
  dWh += -3.061 * 0.000
  dh_next = -2.449

===== UPDATE =====
Wx = 0.560
Wh = 0.807
Wy = 1.111
